In [15]:
import csv
import json

def read_csv(file_path):
    sentences = []
    current_sentence = []
    entities = []
    offset = 0
    current_sent_idx = -1
    current_entity = None
    current_entity_start = None

    label_map = {
        0: None,
        1: 'PERSON', 2: 'PERSON',
        3: 'LOC', 4: 'LOC',
        5: 'ORG', 6: 'ORG'
    }
    
    with open(file_path, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            word, label, sent_idx = row['gold_token'], int(row['gold_label']), int(row['sent_idx'])
            
            if sent_idx != current_sent_idx:  # New sentence
                if current_sentence:
                    sentences.append((" ".join(current_sentence), {"entities": entities}))
                current_sentence = []
                entities = []
                offset = 0
                current_sent_idx = sent_idx
                current_entity = None
                current_entity_start = None
            
            entity_type = label_map.get(label)
            if entity_type:
                if label % 2 == 1:  # B label
                    if current_entity:
                        entities.append((current_entity_start, offset - 1, current_entity))
                    current_entity = entity_type
                    current_entity_start = offset
                elif label % 2 == 0 and entity_type == current_entity:  # I label, continuing entity
                    pass  # Just continue extending the entity
                else:  # Unexpected case, close previous entity
                    if current_entity:
                        entities.append((current_entity_start, offset - 1, current_entity))
                    current_entity = None
                    current_entity_start = None
            else:
                if current_entity:
                    entities.append((current_entity_start, offset - 1, current_entity))
                current_entity = None
                current_entity_start = None
            
            current_sentence.append(word)
            offset += len(word) + 1  # +1 for space
    
    if current_sentence:  # Last sentence
        if current_entity:
            entities.append((current_entity_start, offset - 1, current_entity))
        sentences.append((" ".join(current_sentence), {"entities": entities}))
    
    return sentences

# Uso del script
file_path = "finer_ord_validation.csv"
data = read_csv(file_path)

# Guardar en un archivo JSON
output_file = "finer_validation_data.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"Datos guardados en {output_file}")



Datos guardados en finer_validation_data.json
